In [1]:
# ============================================================
# IMPORTS
# ============================================================
import os
import cv2
import copy
import time
import random
import warnings
import re
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.transforms import RandomErasing
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix, classification_report
)
from tqdm import tqdm

import matplotlib.pyplot as plt
from skimage.segmentation import mark_boundaries

# ---------- SHAP and LIME ----------
import shap
from lime import lime_image

warnings.filterwarnings('ignore')


# ============================================================
# REPRODUCIBILITY
# ============================================================
def set_deterministic(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_deterministic(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# ============================================================
# MIXUP
# ============================================================
class MixUp:
    def __init__(self, alpha=0.2):
        self.alpha = alpha

    def __call__(self, images, labels):
        if self.alpha > 0:
            lam = np.random.beta(self.alpha, self.alpha)
        else:
            lam = 1.0
        batch_size = images.size(0)
        index = torch.randperm(batch_size)
        mixed = lam * images + (1 - lam) * images[index, :]
        labels_a, labels_b = labels, labels[index]
        return mixed, labels_a, labels_b, lam


# ============================================================
# DATASETS
# ============================================================
class OriginalDataset(Dataset):
    """Returns transformed tensors."""
    def __init__(self, image_paths, labels, transform=None, target_size=(256, 256)):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.target_size = target_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        if img is None:
            img = np.zeros((*self.target_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, self.target_size, interpolation=cv2.INTER_AREA)
        img = Image.fromarray(img)
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)


class PILDataset(Dataset):
    """Returns PIL images (used for TTA and LIME)."""
    def __init__(self, image_paths, labels, target_size=(256, 256)):
        self.image_paths = image_paths
        self.labels = labels
        self.target_size = target_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        if img is None:
            img = np.zeros((*self.target_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, self.target_size, interpolation=cv2.INTER_AREA)
        return Image.fromarray(img), torch.tensor(self.labels[idx], dtype=torch.long)


def pil_collate(batch):
    images, labels = zip(*batch)
    return list(images), torch.tensor(labels)


# ============================================================
# MODEL FACTORY (with ablation flag)
# ============================================================
def create_resnet18_ablation(use_dropout=True, num_classes=2, pretrained=True):
    model = models.resnet18(pretrained=pretrained)
    in_features = model.fc.in_features
    if use_dropout:
        model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, num_classes)
        )
    else:
        model.fc = nn.Linear(in_features, num_classes)
    return model.to(device)


# ============================================================
# SIMPLE EVAL (no TTA) — used during training
# ============================================================
def evaluate_simple(model, loader, criterion, device):
    model.eval()
    all_preds, all_targets = [], []
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
    acc = accuracy_score(all_targets, all_preds)
    avg_val_loss = val_loss / len(loader.dataset)
    return acc, avg_val_loss, all_preds, all_targets


# ============================================================
# TRAINING LOOP (with ablation flags)
# ============================================================
def train_model(train_loader, val_loader,
                model_name="ResNet18",
                epochs=15, patience=5,
                use_dropout=True, use_mixup=True):
    print(f"\nTraining {model_name} on {device}...")

    model = create_resnet18_ablation(use_dropout=use_dropout,
                                     num_classes=2, pretrained=True)
    for param in model.parameters():
        param.requires_grad = True

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    mixup = MixUp(alpha=0.2) if use_mixup else None

    best_acc = 0.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for images, labels in tqdm(train_loader,
                                   desc=f"{model_name} Epoch {epoch}/{epochs}",
                                   leave=False):
            images = images.to(device)
            labels = labels.to(device)

            if mixup is not None:
                mixed, la, lb, lam = mixup(images, labels)
                mixed = mixed.to(device)
                la, lb = la.to(device), lb.to(device)
                lam_t = torch.tensor(lam, dtype=torch.float).to(device)

                optimizer.zero_grad()
                outputs = model(mixed)
                loss = lam_t * criterion(outputs, la) + (1 - lam_t) * criterion(outputs, lb)
            else:
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item() * images.size(0)

        val_acc, val_loss, _, _ = evaluate_simple(model, val_loader,
                                                  criterion, device)
        print(f"Epoch {epoch}: Train Loss: "
              f"{running_loss/len(train_loader.dataset):.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    return model, best_acc


# ============================================================
# STANDARD EVAL (no TTA) — returns metrics dict for ablation table
# ============================================================
def evaluate_standard(model, loader, device):
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    n = len(loader.dataset)

    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    if device.type == 'cuda':
        torch.cuda.synchronize()
    total_time = time.perf_counter() - start
    inference_ms = total_time / n * 1000

    targets_np = np.array(all_targets)
    preds_np   = np.array(all_preds)
    probs_np   = np.array(all_probs)

    precision, recall, f1, _ = precision_recall_fscore_support(
        targets_np, preds_np, average=None, zero_division=0)
    macro_p, macro_r, macro_f = precision_recall_fscore_support(
        targets_np, preds_np, average='macro', zero_division=0)[:3]
    weighted_p, weighted_r, weighted_f = precision_recall_fscore_support(
        targets_np, preds_np, average='weighted', zero_division=0)[:3]

    total_params = sum(p.numel() for p in model.parameters())

    return {
        'accuracy': accuracy_score(targets_np, preds_np),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f,
        'weighted_precision': weighted_p,
        'weighted_recall': weighted_r,
        'weighted_f1': weighted_f,
        'pos_precision': precision[1] if len(precision) > 1 else None,
        'pos_recall': recall[1],
        'pos_f1': f1[1],
        'auc': roc_auc_score(targets_np, probs_np[:, 1]),
        'confusion_matrix': confusion_matrix(targets_np, preds_np),
        'total_params': total_params,
        'model_size_mb': total_params * 4 / (1024 * 1024),
        'inference_ms': inference_ms,
        'predictions': preds_np,
        'targets': targets_np,
        'probabilities': probs_np,
    }


# ============================================================
# TTA EVAL
# ============================================================
def evaluate_with_tta(model, pil_loader, device, n_augments=5,
                      model_name="Model"):
    model.eval()
    all_targets, all_preds, all_probs = [], [], []
    inference_times = []

    tta_transforms = [
        transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
        transforms.Compose([
            transforms.RandomHorizontalFlip(p=1.0),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
        transforms.Compose([
            transforms.RandomVerticalFlip(p=1.0),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
        transforms.Compose([
            transforms.RandomHorizontalFlip(p=1.0),
            transforms.RandomVerticalFlip(p=1.0),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
        transforms.Compose([
            transforms.RandomRotation(degrees=15),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
    ]
    tta_transforms = tta_transforms[:n_augments]
    n_aug = len(tta_transforms)

    with torch.no_grad():
        for images, targets in tqdm(pil_loader,
                                    desc=f"TTA eval {model_name}",
                                    leave=False):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start = time.perf_counter()

            tta_batch = []
            for tf in tta_transforms:
                aug = torch.stack([tf(img) for img in images])
                tta_batch.append(aug)
            tta_images = torch.cat(tta_batch, dim=0).to(device)
            outputs = model(tta_images)

            if device.type == 'cuda':
                torch.cuda.synchronize()
            elapsed = (time.perf_counter() - start) * 1000 / len(images)
            inference_times.append(elapsed)

            batch_size = len(images)
            outputs = outputs.view(n_aug, batch_size, -1)
            probs = torch.softmax(outputs, dim=2)
            avg_probs = probs.mean(dim=0)
            _, preds = torch.max(avg_probs, dim=1)

            all_targets.extend(targets.numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(avg_probs.cpu().numpy())

    targets_np = np.array(all_targets)
    preds_np   = np.array(all_preds)
    probs_np   = np.array(all_probs)

    precision, recall, f1, _ = precision_recall_fscore_support(
        targets_np, preds_np, average=None, zero_division=0)
    macro_p, macro_r, macro_f = precision_recall_fscore_support(
        targets_np, preds_np, average='macro', zero_division=0)[:3]
    weighted_p, weighted_r, weighted_f = precision_recall_fscore_support(
        targets_np, preds_np, average='weighted', zero_division=0)[:3]

    total_params = sum(p.numel() for p in model.parameters())

    return {
        'accuracy': accuracy_score(targets_np, preds_np),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f,
        'weighted_precision': weighted_p,
        'weighted_recall': weighted_r,
        'weighted_f1': weighted_f,
        'pos_precision': precision[1] if len(precision) > 1 else None,
        'pos_recall': recall[1],
        'pos_f1': f1[1],
        'auc': roc_auc_score(targets_np, probs_np[:, 1]),
        'confusion_matrix': confusion_matrix(targets_np, preds_np),
        'total_params': total_params,
        'model_size_mb': total_params * 4 / (1024 * 1024),
        'inference_ms': float(np.mean(inference_times)),
        'predictions': preds_np,
        'targets': targets_np,
        'probabilities': probs_np,
    }


# ============================================================
# GRAD-CAM
# ============================================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        def forward_hook(module, inp, out):
            self.activations = out.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        found = False
        for name, module in model.named_modules():
            if name == target_layer:
                module.register_forward_hook(forward_hook)
                module.register_full_backward_hook(backward_hook)   # FIX
                found = True
                break
        if not found:
            raise ValueError(f"Layer '{target_layer}' not found.")

    def generate_cam(self, input_image, target_class=None):
        self.model.eval()
        output = self.model(input_image)
        if target_class is None:
            target_class = torch.argmax(output, dim=1).item()

        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1
        output.backward(gradient=one_hot, retain_graph=True)

        gradients = self.gradients
        activations = self.activations
        weights = torch.mean(gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam = F.interpolate(cam, size=input_image.shape[2:],
                            mode='bilinear', align_corners=False)
        return cam.squeeze().cpu().numpy(), target_class


def visualize_gradcam(model, image_tensor, label, device,
                      target_layer='layer4', save_path=None):
    """image_tensor: (1,3,H,W) already normalized, on device."""
    grad_cam = GradCAM(model, target_layer)
    heatmap, pred_class = grad_cam.generate_cam(image_tensor)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
    img = image_tensor * std + mean
    img = img.squeeze().cpu().permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)

    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized),
                                        cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB) / 255.0
    overlay = np.clip(0.5 * img + 0.5 * heatmap_colored, 0, 1)

    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.axis('off')
    plt.title(f'Grad-CAM | true={label} | pred={pred_class}')
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()


# ============================================================
# SHAP
# ============================================================
def explain_with_shap(model, train_loader, test_images, test_labels,
                      device, num_background=50, class_idx=None):
    num_images = min(5, len(test_images))
    if num_images == 0:
        print("No test images to explain.")
        return

    # Collect background samples
    background = []
    with torch.no_grad():
        for images, _ in train_loader:
            background.append(images)
            if sum(b.shape[0] for b in background) >= num_background:
                break
    background = torch.cat(background, dim=0)[:num_background].cpu()

    model_cpu = model.to('cpu').eval()
    test_images_cpu = test_images[:num_images].cpu()
    test_labels_np  = test_labels[:num_images].cpu().numpy()

    explainer = shap.GradientExplainer(model_cpu, background)
    shap_values = explainer.shap_values(test_images_cpu)   # list of (N,H,W,C)

    # Denormalize images
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    imgs_denorm = torch.clamp(test_images_cpu * std + mean, 0, 1)
    imgs_np = imgs_denorm.numpy().transpose(0, 2, 3, 1)   # (N,H,W,3)

    # Select which class to plot per image
    if class_idx is None:
        # Pick the true class for each image and stack -> (N,H,W,C)
        chosen = np.stack(
            [shap_values[test_labels_np[i]][i] for i in range(num_images)],
            axis=0
        )
    else:
        chosen = np.stack(
            [shap_values[class_idx][i] for i in range(num_images)],
            axis=0
        )

    max_abs = np.max(np.abs(chosen))
    vmax = max_abs * 0.5 if max_abs > 0 else 1e-6
    print(f"Max |SHAP| = {max_abs:.6f}; using vmax = {vmax:.6f}")

    shap.image_plot([chosen], imgs_np, show=False, vmax=vmax)
    plt.show()

    # Restore model
    model.to(device)
    return shap_values


# ============================================================
# LIME
# ============================================================
def explain_with_lime(model, pil_loader, device, num_samples=100,
                      num_images=5, target_size=(256, 256)):
    model_cpu = model.to('cpu').eval()

    def predict(images_np):
        """images_np: list of (H,W,3) float in [0,1]."""
        tf = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(target_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        batch = torch.stack([tf(img) for img in images_np])
        with torch.no_grad():
            out = model_cpu(batch)
        return torch.softmax(out, dim=1).numpy()

    images, labels = next(iter(pil_loader))
    explainer = lime_image.LimeImageExplainer()

    for idx in range(min(num_images, len(images))):
        img_np = np.array(images[idx].resize(target_size)) / 255.0
        explanation = explainer.explain_instance(
            img_np, predict,
            top_labels=2, hide_color=0, num_samples=num_samples
        )
        temp, mask = explanation.get_image_and_mask(
            explanation.top_labels[0],
            positive_only=True, num_features=10, hide_rest=False
        )
        plt.figure(figsize=(5, 5))
        plt.imshow(mark_boundaries(temp, mask))
        plt.axis('off')
        plt.title(f"LIME | true={labels[idx].item()}")
        plt.show()

    model.to(device)


# ============================================================
# MAIN — DATA + PATIENT-AWARE SPLIT
# ============================================================
set_deterministic(42)

base_path = '/kaggle/input/datasets/iarunava/cell-images-for-detecting-malaria/cell_images'
categories = ['Parasitized', 'Uninfected']   # label 0 = Parasitized, 1 = Uninfected

image_paths, labels = [], []
for cat_idx, cat in enumerate(categories):
    folder = os.path.join(base_path, cat)
    exts = ('.png', '.jpg', '.jpeg')
    files = sorted([os.path.join(folder, f)
                    for f in os.listdir(folder)
                    if f.lower().endswith(exts)])
    image_paths.extend(files)
    labels.extend([cat_idx] * len(files))

path_to_label = dict(zip(image_paths, labels))

patient_to_images = defaultdict(list)
for path in image_paths:
    match = re.search(r'P(\d+)', os.path.basename(path))
    pid = f"P{match.group(1)}" if match else os.path.basename(path).split('_')[0]
    patient_to_images[pid].append(path)

unique_patients = sorted(patient_to_images.keys())

patient_majority_label = {}
for pid in unique_patients:
    lbls = [path_to_label[p] for p in patient_to_images[pid]]
    counts = Counter(lbls)
    max_count = max(counts.values())
    patient_majority_label[pid] = min([c for c, n in counts.items() if n == max_count])

patient_labels = [patient_majority_label[pid] for pid in unique_patients]

train_patients, test_patients = train_test_split(
    unique_patients, test_size=0.20, random_state=42, stratify=patient_labels
)
train_set, test_set = set(train_patients), set(test_patients)

train_paths, train_labels = [], []
test_paths, test_labels = [], []

for pid in unique_patients:
    for p in patient_to_images[pid]:
        if pid in train_set:
            train_paths.append(p); train_labels.append(path_to_label[p])
        else:
            test_paths.append(p); test_labels.append(path_to_label[p])

print(f"Total patients: {len(unique_patients)}")
print(f"Train: {len(train_set)} patients, {len(train_paths)} cells")
print(f"Test : {len(test_set)} patients, {len(test_paths)} cells")
print(f"Train classes: {Counter(train_labels)}")
print(f"Test  classes: {Counter(test_labels)}")


# ============================================================
# TRANSFORMS + DATALOADERS
# ============================================================
target_size = (256, 256)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    RandomErasing(p=0.2, scale=(0.02, 0.1))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = OriginalDataset(train_paths, train_labels, train_transform, target_size)
test_ds  = OriginalDataset(test_paths,  test_labels,  test_transform,  target_size)
pil_ds   = PILDataset(test_paths, test_labels, target_size)

batch_size = 64
train_loader = DataLoader(train_ds, batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)
pil_loader   = DataLoader(pil_ds, batch_size, shuffle=False,
                          num_workers=4, collate_fn=pil_collate)


# ============================================================
# ABLATION — 4 CONFIGS
# ============================================================
ablation_configs = [
    {'name': '(1) ResNet18 baseline', 'use_dropout': False, 'use_tta': False},
    {'name': '(2) + Dropout(0.5)',    'use_dropout': True,  'use_tta': False},
    {'name': '(3) + TTA',             'use_dropout': False, 'use_tta': True},
    {'name': '(4) + Dropout + TTA',   'use_dropout': True,  'use_tta': True},
]

ablation_results = {}
ablation_states  = {}

for cfg in ablation_configs:
    print("\n" + "=" * 70)
    print(f"ABLATION: {cfg['name']}")
    print("=" * 70)

    set_deterministic(42)   # reset seed for fair comparison

    model, best_acc = train_model(
        train_loader, test_loader,
        model_name=cfg['name'],
        epochs=15, patience=5,
        use_dropout=cfg['use_dropout'],
        use_mixup=True
    )
    print(f"Best validation accuracy: {best_acc*100:.2f}%")

    if cfg['use_tta']:
        metrics = evaluate_with_tta(model, pil_loader, device,
                                    n_augments=5, model_name=cfg['name'])
    else:
        metrics = evaluate_standard(model, test_loader, device)

    ablation_results[cfg['name']] = metrics
    ablation_states[cfg['name']]  = copy.deepcopy(model.state_dict())

    print(f"Test accuracy  : {metrics['accuracy']*100:.2f}%")
    print(f"Test precision : {metrics['pos_precision']:.4f}")
    print(f"Test recall    : {metrics['pos_recall']:.4f}")
    print(f"Test F1        : {metrics['pos_f1']:.4f}")
    print(f"Test AUC       : {metrics['auc']:.4f}")
    print(f"Inference time : {metrics['inference_ms']:.2f} ms/img")
    print(f"Confusion Matrix:\n{metrics['confusion_matrix']}")

    safe = cfg['name'].replace(' ', '_').replace('(', '').replace(')', '')
    torch.save(model.state_dict(), f'ablation_{safe}.pth')
    print(f"Saved: ablation_{safe}.pth")


# ============================================================
# SUMMARY + COMPONENT CONTRIBUTIONS
# ============================================================
print("\n" + "=" * 90)
print("ABLATION SUMMARY (Patient-Level Split, Uniform Training Protocol)")
print("=" * 90)
print(f"{'Configuration':<30} {'Acc%':>7} {'Prec':>7} {'Rec':>7} "
      f"{'F1':>7} {'AUC':>7} {'Time (ms)':>11}")
print("-" * 90)
for name, m in ablation_results.items():
    print(f"{name:<30} {m['accuracy']*100:>7.2f} {m['pos_precision']:>7.4f} "
          f"{m['pos_recall']:>7.4f} {m['pos_f1']:>7.4f} {m['auc']:>7.4f} "
          f"{m['inference_ms']:>11.2f}")
print("=" * 90)

base_acc = ablation_results['(1) ResNet18 baseline']['accuracy']
drop_acc = ablation_results['(2) + Dropout(0.5)']['accuracy']
tta_acc  = ablation_results['(3) + TTA']['accuracy']
both_acc = ablation_results['(4) + Dropout + TTA']['accuracy']

print("\n" + "=" * 60)
print("COMPONENT CONTRIBUTIONS (accuracy gains)")
print("=" * 60)
print(f"Dropout alone        : {(drop_acc - base_acc)*100:+.2f} pp")
print(f"TTA alone            : {(tta_acc  - base_acc)*100:+.2f} pp")
print(f"Dropout + TTA (full) : {(both_acc - base_acc)*100:+.2f} pp")
print(f"Interaction          : "
      f"{((both_acc - base_acc) - (drop_acc - base_acc) - (tta_acc - base_acc))*100:+.2f} pp")
print("=" * 60)


# ============================================================
# BEST CONFIG
# ============================================================
best_cfg_name = max(ablation_results, key=lambda k: ablation_results[k]['accuracy'])
best_metrics  = ablation_results[best_cfg_name]
best_cfg = next(c for c in ablation_configs if c['name'] == best_cfg_name)

print(f"\nBest configuration: {best_cfg_name}")
print(f"Accuracy: {best_metrics['accuracy']*100:.2f}%")
print("\nClassification Report:")
print(classification_report(
    best_metrics['targets'], best_metrics['predictions'],
    target_names=['Parasitized', 'Uninfected']))


# ============================================================
# REBUILD BEST MODEL FOR EXPLAINABILITY
# ============================================================
best_model = create_resnet18_ablation(use_dropout=best_cfg['use_dropout'],
                                      num_classes=2, pretrained=False)
best_model.load_state_dict(ablation_states[best_cfg_name])
best_model = best_model.to(device).eval()


# ============================================================
# EXPLAINABILITY ON BEST MODEL
# ============================================================
print("\n" + "=" * 60)
print(f"EXPLAINABILITY ON BEST CONFIG: {best_cfg_name}")
print("=" * 60)


print("\nDone.")

Using device: cuda
Total patients: 200
Train: 160 patients, 22321 cells
Test : 40 patients, 5237 cells
Train classes: Counter({0: 11303, 1: 11018})
Test  classes: Counter({1: 2761, 0: 2476})

ABLATION: (1) ResNet18 baseline

Training (1) ResNet18 baseline on cuda...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 258MB/s]


Epoch 1: Train Loss: 0.3756 | Val Loss: 0.2784 | Val Acc: 96.31%


Epoch 2: Train Loss: 0.3536 | Val Loss: 0.2745 | Val Acc: 96.87%


Epoch 3: Train Loss: 0.3511 | Val Loss: 0.2567 | Val Acc: 97.40%


...

Epoch 12: Train Loss: 0.3290 | Val Loss: 0.2561 | Val Acc: 97.84%


Epoch 13: Train Loss: 0.3317 | Val Loss: 0.2478 | Val Acc: 97.82%


Epoch 14: Train Loss: 0.3350 | Val Loss: 0.2692 | Val Acc: 97.88%


Epoch 15: Train Loss: 0.3307 | Val Loss: 0.2495 | Val Acc: 97.56%
Best validation accuracy: 97.88%


Test accuracy  : 97.94%
Test precision : 0.9787
Test recall    : 0.9823
Test F1        : 0.9805
Test AUC       : 0.9965
Inference time : 13.36 ms/img
Confusion Matrix:
[[2417   59]
 [  49 2712]]
Saved: ablation_4_+_Dropout_+_TTA.pth

ABLATION SUMMARY (Patient-Level Split, Uniform Training Protocol)
Configuration                     Acc%    Prec     Rec      F1     AUC   Time (ms)
------------------------------------------------------------------------------------------
(1) ResNet18 baseline            97.94  0.9808  0.9801  0.9804  0.9965        1.67
(2) + Dropout(0.5)               97.88  0.9783  0.9815  0.9799  0.9962        1.70
(3) + TTA                        97.96  0.9798  0.9815  0.9806  0.9966       13.32
(4) + Dropout + TTA              97.94  0.9787  0.9823  0.9805  0.9965       13.36

COMPONENT CONTRIBUTIONS (accuracy gains)
Dropout alone        : -0.06 pp
TTA alone            : +0.02 pp
Dropout + TTA (full) : +0.00 pp
Interaction          : +0.04 pp

Best configuration: (3)